In [4]:
#from aauwss_audit.ipynb
def check_epoch_quality(epoch_signals, label, thresholds):
    if label not in ['R','REM','N1','N2','N3','W','Wake']: return (False, f'Label is {label}')

    for k, signal in epoch_signals.items():
        signal_arr = np.asarray(signal, dtype=float)
        if len(signal_arr) == 0:
            return (False, f'{k} is empty')
        missing_ratio = np.isnan(signal_arr).mean()
        if missing_ratio > thresholds['max_missing_ratio']:
            return (False, f'{k} has low coverage({round(missing_ratio*100, 1)}%)')

    temp = np.asarray(epoch_signals['TEMP'], dtype=float)
    temp_valid = temp[~np.isnan(temp)]
    if len(temp_valid) > 0:
        temp_off_wrist_ratio = ((temp_valid < thresholds['temp_min']) | (temp_valid > thresholds['temp_max'])).mean()
        if temp_off_wrist_ratio > thresholds['temp_off_wrist_max_ratio']:
            return (False, 'Band not on wrist(TEMP off bounds)')

    eda = np.asarray(epoch_signals['EDA'], dtype=float)
    eda_valid = eda[~np.isnan(eda)]
    if len(eda_valid) > 0:
        if (eda_valid < thresholds['eda_min']).all():
            return (False, 'Band not on wrist(EDA too low)')
    
    bvp = np.asarray(epoch_signals['BVP'], dtype=float)
    bvp_valid = bvp[~np.isnan(bvp)]
    if len(bvp_valid) > 0 and np.var(bvp_valid) < thresholds['bvp_min_var']:
        return (False, 'Flat BVP')
    
    hr = np.asarray(epoch_signals['HR'], dtype=float)
    hr_valid = hr[~np.isnan(hr)]
    if len(hr_valid) > 0:
        hr_out_of_bounds = ((hr_valid < thresholds['hr_min']) | (hr_valid > thresholds['hr_max'])).mean()
        if hr_out_of_bounds > 0.0:
            return (False, 'HR out of bounds')

    return (True, 'OK')

def read_raw_slice(file_path, start_row, n_rows):
    if pd.isna(file_path) or not Path(file_path).exists() or pd.isna(start_row):
        return np.array([])
    return pd.read_csv(file_path, skiprows=int(start_row)+2, nrows=int(n_rows), header=None)[0].to_numpy()


In [19]:
import numpy as np

def HR_features_one_epoch(hr, freq=1):
    hr_arr = np.asarray(hr, dtype=float)
    hr_valid = hr_arr[~np.isnan(hr_arr)]
    
    if len(hr_valid) == 0:
        return {'HR_mean': 0.0, 'HR_std': 0.0, 'HR_min': 0.0, 'HR_max': 0.0,'beat_count': 0.0, 'mean_ibi': 0.0, 
                'sdnn': 0.0, 'rmssd': 0.0,'pnn50': 0.0, 'dominant_freq': 0.0}

    mean_hr, std_hr = np.mean(hr_valid), np.std(hr_valid)
    min_hr, max_hr = np.min(hr_valid), np.max(hr_valid)
    beat_count = mean_hr*30.0/60.0

    safe_hr = np.where(hr_valid <= 0, np.nan, hr_valid)
    ibi = 60000.0/safe_hr
    ibi = ibi[~np.isnan(ibi)]
    
    if len(ibi) > 0:
        mean_ibi, sdnn = np.mean(ibi), np.std(ibi)
        ibi_diff = np.diff(ibi)
        rmssd = np.sqrt(np.mean(ibi_diff**2)) if len(ibi_diff) > 0 else 0.0
        pnn50 = (np.sum(np.abs(ibi_diff) > 50)/len(ibi_diff)*100.0) if len(ibi_diff) > 0 else 0.0
    else:
        mean_ibi, sdnn, rmssd, pnn50 = 0.0, 0.0, 0.0, 0.0

    n_samples = len(hr_valid)
    if n_samples > 1 and std_hr > 0:
        hr_demeaned = hr_valid-mean_hr
        windowed_hr = hr_demeaned*np.hanning(n_samples)
        
        n_fft = 256
        fft_vals = np.fft.rfft(windowed_hr, n=n_fft)
        fft_freqs = np.fft.rfftfreq(n_fft, d=1.0/freq)
        fft_magnitudes = np.abs(fft_vals)
        fft_magnitudes[0] = 0.0  
        dominant_freq = fft_freqs[np.argmax(fft_magnitudes)]
    else:
        dominant_freq = 0.0

    return {'HR_mean': mean_hr, 'HR_std': std_hr, 'HR_min': min_hr, 'HR_max': max_hr, 'beat_count': beat_count, 
            'mean_ibi': mean_ibi, 'sdnn': sdnn, 'rmssd': rmssd, 'pnn50': pnn50, 'dominant_freq': dominant_freq}

def ACC_features_one_epoch(acc_matrix, prev_angle=None, fs=64):
    if acc_matrix.size == 0 or acc_matrix.shape[0] == 0:
        features = {'mag_mean': 0.0, 'mag_std': 0.0, 'mag_max': 0.0, 'jerk_mean': 0.0, 'sd_x': 0.0, 
                    'sd_y': 0.0, 'sd_z': 0.0, 'share_away_1g': 0.0, 'angle_change': 0.0}
        return features, prev_angle
    x, y, z = acc_matrix[:, 0], acc_matrix[:, 1], acc_matrix[:, 2]
    
    magnitude = np.sqrt(x**2+y**2+z**2)
    mag_mean = np.mean(magnitude)
    mag_std = np.std(magnitude)
    mag_max = np.max(magnitude)
    
    dt = 1.0/fs
    jerk_x = np.diff(x)/dt
    jerk_y = np.diff(y)/dt
    jerk_z = np.diff(z)/dt
    
    jerk_mag = np.sqrt(jerk_x**2+jerk_y**2+jerk_z**2)
    mag_jerk_mean = np.mean(jerk_mag) if len(jerk_mag) > 0 else 0.0
    
    sd_x = np.std(x)
    sd_y = np.std(y)
    sd_z = np.std(z)
    
    tolerance = 0.1
    samples_away = np.sum(np.abs(magnitude-1.0) > tolerance)
    share_away_1g = samples_away/len(magnitude) if len(magnitude) > 0 else 0.0
    
    current_angles = np.arctan2(y, np.sqrt(x**2+z**2))*(180.0/np.pi)
    current_angle_mean = np.mean(current_angles)
    
    if prev_angle is not None:
        angle_change = np.abs(current_angle_mean-prev_angle)
    else:
        angle_change = 0.0
        
    features = {'mag_mean': mag_mean,'mag_std': mag_std,'mag_max': mag_max,'jerk_mean': mag_jerk_mean,
                'sd_x': sd_x,'sd_y': sd_y,'sd_z': sd_z,'share_away_1g': share_away_1g,'angle_change': angle_change}
    return features, current_angle_mean

def EDA_features_one_epoch(eda, fs=64):
    eda = np.array(eda)
    n_samples = len(eda)

    if n_samples == 0:
        return {'eda_mean': 0.0, 'eda_std': 0.0, 'eda_min': 0.0, 'eda_max': 0.0, 'eda_slope': 0.0, 'eda_num_peaks': 0.0}
    
    mean = np.mean(eda)
    std = np.std(eda)
    mini = np.min(eda)
    maxi = np.max(eda)
    
    if n_samples > 1:
        time_axis = np.arange(n_samples)/fs
        slope, _ = np.polyfit(time_axis, eda, 1)
    else:
        slope = 0.0
        
    num_peaks = 0
    prominence_threshold = 0.02     
    for i in range(1, n_samples-1):
        if eda[i] > eda[i-1] and eda[i] > eda[i+1]:
            left_min = eda[i]
            for l in range(i-1, -1, -1):
                if eda[l] < left_min: left_min = eda[l]
                if l > 0 and eda[l-1] > eda[l]: break
                
            right_min = eda[i]
            for r in range(i+1, n_samples):
                if eda[r] < right_min: right_min = eda[r]
                if r < n_samples-1 and eda[r+1] > eda[r]: break
                
            prominence = eda[i]-max(left_min, right_min)
            if prominence >= prominence_threshold:
                num_peaks += 1

    return {'eda_mean': mean,'eda_std': std,'eda_min': mini,'eda_max': maxi,'eda_slope': slope,'eda_num_peaks': num_peaks}


def TEMP_features_one_epoch(temp, fs=64):
    if len(temp) == 0:
        return {'temp_mean': 0.0, 'temp_std': 0.0, 'temp_min': 0.0, 'temp_max': 0.0, 'temp_slope': 0.0}
    temp = np.array(temp)
    mean = np.mean(temp)
    std = np.std(temp)
    mini = np.min(temp)
    maxi = np.max(temp)
    
    time_axis = np.arange(len(temp))/fs
    slope, _ = np.polyfit(time_axis, temp, 1)
    
    return {'temp_mean': mean,'temp_std': std,'temp_min': mini,'temp_max': maxi,'temp_slope': slope}

def add_context_features(df, feature_cols, window=5):
    df_context = df.copy()
    grouped = df_context.groupby('participant')
    
    for col in feature_cols:
        rolling_mean = grouped[col].transform(lambda x: x.rolling(window=2*window+1, center=True, min_periods=1).mean())
        df_context[f'{col}_context_mean'] = rolling_mean
    return df_context

In [8]:
import numpy as np
import pandas as pd

def downsample_dreamt_to_native(df_64hz):
    total_rows = len(df_64hz)
    bvp_native = df_64hz['BVP'].values
    
    def block_avg(series_name, factor):
        valid_len = (total_rows//factor)*factor
        vals = df_64hz[series_name].values[:valid_len]
        return vals.reshape(-1, factor).mean(axis=1)

    acc_x_native = block_avg('ACC_X', 2)/64.0
    acc_y_native = block_avg('ACC_Y', 2)/64.0
    acc_z_native = block_avg('ACC_Z', 2)/64.0
    
    eda_native = block_avg('EDA', 16)
    temp_native = block_avg('TEMP', 16)
    hr_native = block_avg('HR', 64)
    
    native_signals = {'BVP': bvp_native,'ACC_32Hz': np.column_stack([acc_x_native, acc_y_native, acc_z_native]),
                      'EDA_4Hz': eda_native,'TEMP_4Hz': temp_native,'HR_1Hz': hr_native}
    
    return native_signals


In [ ]:
import os
import gc
from pathlib import Path
import yaml
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name != 'project' else Path.cwd().resolve()
with open(ROOT/'config.yaml', 'r') as file:
    config = yaml.safe_load(file)

output_dir = ROOT/'data'/'interim'
output_dir.mkdir(parents=True, exist_ok=True)
inventory_dir = ROOT/'results'/'inventory'

DRY_RUN = False  
DRY_RUN_LIMIT = 5

ENABLE_CONTEXT_FEATURES = True

print(f"Starting DREAMT Feature Extraction")
dreamt_data_path = ROOT/config['dreamt']['path']/'data_64Hz'
dreamt_files = sorted([f for f in os.listdir(dreamt_data_path) if f.endswith('.csv')])

if DRY_RUN:
    dreamt_files = dreamt_files[:DRY_RUN_LIMIT]

dreamt_rows = []
for idx, file in enumerate(dreamt_files):
    participant_id = file[:4]
    df_raw = pd.read_csv(dreamt_data_path/file)
    
    native = downsample_dreamt_to_native(df_raw)
    
    non_p_indices = df_raw.index[df_raw['Sleep_Stage'] != 'P'].tolist()
    if not non_p_indices:
        continue
    
    first_scored_row = non_p_indices[0]
    total_rows = len(df_raw)
    epoch_num = 0
    prev_angle = None

    for start_64 in range(first_scored_row, total_rows-1920+1, 1920):
        end_64 = start_64+1920
        label = df_raw['Sleep_Stage'].iloc[start_64]

        start_32, end_32 = start_64//2, end_64//2
        start_4, end_4 = start_64//16, end_64//16
        start_1, end_1 = start_64//64, end_64//64

        epoch_signals = {'BVP': native['BVP'][start_64:end_64],'ACC_X': native['ACC_32Hz'][start_32:end_32, 0],'ACC_Y': native['ACC_32Hz'][start_32:end_32, 1],'ACC_Z': native['ACC_32Hz'][start_32:end_32, 2],'EDA': native['EDA_4Hz'][start_4:end_4],
                         'TEMP': native['TEMP_4Hz'][start_4:end_4],'HR': native['HR_1Hz'][start_1:end_1]}

        usable, reason = check_epoch_quality(epoch_signals, label, config['thresholds'])
        
        f_hr = HR_features_one_epoch(epoch_signals['HR'], freq=1)
        f_acc, prev_angle = ACC_features_one_epoch(epoch_signals['ACC_X'], epoch_signals['ACC_Y'], epoch_signals['ACC_Z'], prev_angle=prev_angle, fs=32)
        f_eda = EDA_features_one_epoch(epoch_signals['EDA'], fs=4)
        f_temp = TEMP_features_one_epoch(epoch_signals['TEMP'], fs=4)

        dreamt_rows.append({'participant': participant_id,'epoch_number': epoch_num,'label': label,'usable': usable,'reason': reason,**f_hr, **f_acc, **f_eda, **f_temp})
        epoch_num += 1

    del df_raw, native
    gc.collect()
    print(f"DREAMT [{idx+1}/{len(dreamt_files)}]: Completed {participant_id}")

df_dreamt = pd.DataFrame(dreamt_rows)
feature_cols = [c for c in df_dreamt.columns if c not in ['participant', 'epoch_number', 'label', 'usable', 'reason']]

if ENABLE_CONTEXT_FEATURES:
    df_dreamt = add_context_features(df_dreamt, feature_cols, window=5)

out_dreamt_path = output_dir/('features_dreamt_dryrun.parquet' if DRY_RUN else 'features_dreamt.parquet')
df_dreamt.to_parquet(out_dreamt_path, index=False)
print(f"Done! DREAMT features written to: {out_dreamt_path} (Shape: {df_dreamt.shape})\n")

Starting DREAMT Feature Extraction
DREAMT [1/100]: Completed S002
DREAMT [2/100]: Completed S003
DREAMT [3/100]: Completed S004
DREAMT [4/100]: Completed S005
DREAMT [5/100]: Completed S006
DREAMT [6/100]: Completed S007
DREAMT [7/100]: Completed S008
DREAMT [8/100]: Completed S009
DREAMT [9/100]: Completed S010
DREAMT [10/100]: Completed S011
DREAMT [11/100]: Completed S012
DREAMT [12/100]: Completed S013
DREAMT [13/100]: Completed S014
DREAMT [14/100]: Completed S015
DREAMT [15/100]: Completed S016
DREAMT [16/100]: Completed S017
DREAMT [17/100]: Completed S018
DREAMT [18/100]: Completed S019
DREAMT [19/100]: Completed S020
DREAMT [20/100]: Completed S021
DREAMT [21/100]: Completed S022
DREAMT [22/100]: Completed S023
DREAMT [23/100]: Completed S024
DREAMT [24/100]: Completed S025
DREAMT [25/100]: Completed S026
DREAMT [26/100]: Completed S027
DREAMT [27/100]: Completed S028
DREAMT [28/100]: Completed S029
DREAMT [29/100]: Completed S030
DREAMT [30/100]: Completed S031
DREAMT [31/100

In [20]:
import os
import gc
import csv
from pathlib import Path
import itertools
import yaml
import numpy as np
import pandas as pd

def read_raw_slice_fast(file_path, start_row, n_rows, is_acc=False):
    if pd.isna(file_path) or not Path(file_path).exists() or pd.isna(start_row):
        return np.empty((0, 3)) if is_acc else np.array([])
    
    skip_lines = int(start_row)+2
    
    with open(file_path, 'r', newline='', encoding='utf-8') as f:
        reader = csv.reader(f)
        slice_iterator = itertools.islice(reader, skip_lines, skip_lines+int(n_rows))
        
        if is_acc:
            data = [list(map(float, row)) for row in slice_iterator if row]
            return np.array(data) if data else np.empty((0, 3))
        else:
            data = [float(row[0]) for row in slice_iterator if row]
            return np.array(data) if data else np.array([])

ROOT = Path.cwd().resolve().parent if Path.cwd().name != 'project' else Path.cwd().resolve()
with open(ROOT/'config.yaml', 'r') as file:
    config = yaml.safe_load(file)

output_dir = ROOT/'data'/'interim'
output_dir.mkdir(parents=True, exist_ok=True)
inventory_dir = ROOT/'results'/'inventory'

DRY_RUN = False
DRY_RUN_LIMIT = 1
ENABLE_CONTEXT_FEATURES = True

aauwss_path = ROOT/config['aauwss']['path']
alignment_df = pd.read_csv(inventory_dir/'aauwss_epoch_alignment.csv')

participants = alignment_df['participant'].unique()
if DRY_RUN:
    participants = participants[:DRY_RUN_LIMIT]

aauwss_rows = []

for idx, participant_id in enumerate(participants):
    group = alignment_df[alignment_df['participant'] == participant_id]
    empatica_dir = aauwss_path/'empatica'/participant_id
    prev_angle = None

    for _, epoch_row in group.iterrows():
        segment_file = epoch_row['segment_file']
        
        if pd.isna(segment_file) or str(epoch_row['usable']).lower() == 'false':
            f_hr = HR_features_one_epoch([], freq=1)
            f_acc, prev_angle = ACC_features_one_epoch(np.empty((0, 3)), prev_angle=prev_angle, fs=32)
            f_eda = EDA_features_one_epoch([], fs=4)
            f_temp = TEMP_features_one_epoch([], fs=4)

            aauwss_rows.append({'participant': participant_id, 'epoch_number': epoch_row['epoch_number'],
                                'label': epoch_row['sleep_stage'], 'usable': False, 
                                'reason': str(epoch_row['reason']),**f_hr, **f_acc, **f_eda, **f_temp})
            continue

        seg_dir = list(empatica_dir.glob(f"**/{segment_file}"))[0].parent
        start_64hz = epoch_row['start_row_64hz']

        acc_signal = read_raw_slice_fast(seg_dir/'ACC.csv', start_64hz/2, 960, is_acc=True)/64.0
        
        bvp_signal  = read_raw_slice_fast(seg_dir/'BVP.csv', start_64hz, 1920)
        eda_signal  = read_raw_slice_fast(seg_dir/'EDA.csv', start_64hz/16, 120)
        temp_signal = read_raw_slice_fast(seg_dir/'TEMP.csv', start_64hz/16, 120)
        hr_signal   = read_raw_slice_fast(seg_dir/'HR.csv', start_64hz/64, 30)

        epoch_signals = {'BVP': bvp_signal,'ACC': acc_signal,'EDA': eda_signal,'TEMP': temp_signal,'HR': hr_signal}
        usable, reason = check_epoch_quality(epoch_signals, epoch_row['sleep_stage'], config['thresholds'])
        
        f_hr = HR_features_one_epoch(epoch_signals['HR'], freq=1)
        f_acc, prev_angle = ACC_features_one_epoch(epoch_signals['ACC'], prev_angle=prev_angle, fs=32)
        f_eda = EDA_features_one_epoch(epoch_signals['EDA'], fs=4)
        f_temp = TEMP_features_one_epoch(epoch_signals['TEMP'], fs=4)

        aauwss_rows.append({
            'participant': participant_id, 'epoch_number': epoch_row['epoch_number'], 'label': epoch_row['sleep_stage'],
            'usable': usable, 'reason': reason, **f_hr, **f_acc, **f_eda, **f_temp})

    gc.collect()
    print(f"AAUWSS [{idx+1}/{len(participants)}]: Completed {participant_id}")

df_aauwss = pd.DataFrame(aauwss_rows)
feature_cols = [c for c in df_aauwss.columns if c not in ['participant', 'epoch_number', 'label', 'usable', 'reason']]
if ENABLE_CONTEXT_FEATURES:
    df_aauwss = add_context_features(df_aauwss, feature_cols, window=5)

out_aauwss_path = output_dir/('features_aauwss_dryrun.parquet' if DRY_RUN else 'features_aauwss.parquet')
df_aauwss.to_parquet(out_aauwss_path, index=False)
print(f"AAUWSS features written to: {out_aauwss_path} (Shape: {df_aauwss.shape})")


AAUWSS [1/13]: Completed subject_01
AAUWSS [2/13]: Completed subject_02
AAUWSS [3/13]: Completed subject_03
AAUWSS [4/13]: Completed subject_04
AAUWSS [5/13]: Completed subject_05
AAUWSS [6/13]: Completed subject_06
AAUWSS [7/13]: Completed subject_07
AAUWSS [8/13]: Completed subject_08
AAUWSS [9/13]: Completed subject_09
AAUWSS [10/13]: Completed subject_10
AAUWSS [11/13]: Completed subject_11
AAUWSS [12/13]: Completed subject_12
AAUWSS [13/13]: Completed subject_13
AAUWSS features written to: D:\Adobe\projects\research proj\data\interim\features_aauwss.parquet (Shape: (11460, 65))


In [ ]:
import pandas as pd
from pathlib import Path
import yaml

ROOT = Path.cwd().resolve().parent
with open(ROOT/'config.yaml','r') as file:
    config = yaml.safe_load(file)
inventory_dir = ROOT/'results'/'inventory'
reports_dir = ROOT/'reports'

def generate_epoch_summary(quality_file, save_path):
    quality_df = pd.read_csv(inventory_dir/quality_file)
    usable = quality_df[quality_df['usable']].groupby(by=['participant','sleep_stage']).size()
    usable_df = usable.reset_index().pivot(index='participant',columns='sleep_stage',values=0).fillna(0)
    usable_df.columns = [f'{c}_usable' for c in usable_df.columns]

    excluded_epochs = quality_df[~quality_df['usable']].groupby(by=['participant','reason']).size()
    excluded_df = excluded_epochs.reset_index().pivot(index='participant',columns='reason',values=0).fillna(0)
    final_df = usable_df.merge(excluded_df, how='left',left_index=True, right_index=True).fillna(0)
    final_df.to_csv(save_path,index=True)
    print(f"Saved usable epoch summary of {quality_file} to {save_path}")

generate_epoch_summary('dreamt_quality_audit.csv',inventory_dir/'dreamt_usable_epochs.csv')
generate_epoch_summary('aauwss_quality_audit.csv',inventory_dir/'aauwss_usable_epochs.csv')

Saved usable epoch summary of dreamt_quality_audit.csv to D:\Adobe\projects\research proj\results\inventory\dreamt_usable_epochs.csv
Saved usable epoch summary of aauwss_quality_audit.csv to D:\Adobe\projects\research proj\results\inventory\aauwss_usable_epochs.csv
